# About this notebook ✨

This notebook contains the monolithic implementation of the **Shortest Path Percolation (SPP)** model in Python.  

It is intended for people with little or no experience in programming and in executing commands (*prompts*) in interactive environments.  
The notebook can be run in a Python environment (Google Colab or Anaconda), allowing you to perform simulations and analyses in a practical way.


# 📦 Importing Libraries

In [1]:
# 📚 Core Libraries
import sys
import os
import time
import math
import random

# 📊 Data Handling
import pandas as pd
import numpy as np


### 🎲 Generating Random Numbers (Mersenne Twister 64-bit)

In [2]:
# Random Number Generation (Mersenne Twister 64-bit)
# Placeholder implementation using Python's built-in random
# instead of the full MT64 algorithm.

NN = 312
mt = [0] * NN
mti = 0

def genrand64_int64():
    """
    Generate a 64-bit unsigned random integer.

    In the original C implementation, this would come from the full
    Mersenne Twister 64-bit algorithm. Here, it is implemented as a
    placeholder using Python's `random.getrandbits(64)`.

    Returns
    -------
    int
        A random 64-bit integer.
    """
    return random.getrandbits(64)


def genrand64_real3():
    """
    Generate a random floating-point number in [0, 1).

    In the C reference code, the formula is:
        ((genrand64_int64() >> 12) + 0.5) * (1.0 / 4503599627370496.0)

    Here, as a placeholder, we normalize a 64-bit random integer.

    Returns
    -------
    float
        A random float uniformly distributed in [0, 1).
    """
    return genrand64_int64() / float(1 << 64)


def init_genrand64(seed):
    """
    Initialize the state array for the 64-bit Mersenne Twister.

    Parameters
    ----------
    seed : int
        Initial seed value for the generator.

    Notes
    -----
    This function partially mimics the initialization procedure
    of MT19937-64, but does not fully replicate the algorithm.
    """
    global mt, mti
    mt[0] = seed
    for mti in range(1, NN):
        mt[mti] = (
            6364136223846793005 * (mt[mti - 1] ^ (mt[mti - 1] >> 62)) + mti
        ) & ((1 << 64) - 1)


### 🛠️ Helper Functions (Support Tools)

In [3]:
def find_node(i, j, bond):
    """
    Find the index 'k' such that bond[i][k] == j.

    Parameters
    ----------
    i : int
        Starting node index.
    j : int
        Target node index.
    bond : list
        Adjacency-like structure storing connections.

    Returns
    -------
    int
        Index 'k' if the connection exists, otherwise -1.
    """
    for k in range(1, bond[0][i] + 1):
        if bond[i][k] == j:
            return k
    return -1


def copy_network(N, origin_bond, copy_bond):
    """
    Copy the adjacency structure from one network to another.

    Parameters
    ----------
    N : int
        Number of nodes.
    origin_bond : list
        Original adjacency-like structure.
    copy_bond : list
        Target adjacency-like structure to be filled.
    """
    for i in range(1, N+1):
        degree = origin_bond[0][i]
        copy_bond[0][i] = degree
        copy_bond[i] = origin_bond[i][:degree+1]  # Copy adjacency list


def generate_ERgraph(N, bond, E):
    """
    Generate an Erdős-Rényi random graph with N nodes and E edges.

    Parameters
    ----------
    N : int
        Number of nodes.
    bond : list
        Adjacency-like structure to populate with connections.
    E : int
        Number of edges to add.

    Notes
    -----
    This function does not update an explicit 'edges' list.
    """
    for i in range(1, N+1):
        bond[0][i] = 0

    for _ in range(E):
        n = int(genrand64_real3() * N) + 1
        m = int(genrand64_real3() * N)
        control = 0
        while control == 0:
            m += 1
            if m > N:
                m = 1
            if n != m and find_node(n, m, bond) < 0:
                control = 1
        bond[0][n] += 1
        bond[n].append(m)
        bond[0][m] += 1
        bond[m].append(n)


def clean_time_stamp(update_time, E):
    """
    Reset the time_stamp array to zero.

    Parameters
    ----------
    update_time : list
        Array storing timestamps for edges.
    E : int
        Number of edges.
    """
    for i in range(1, E+1):
        update_time[i] = 0


### 🔍 Breadth-First Search (BFS) and Path Utilities

In [4]:
def simple_bfs(C, N, bond, source, target, vec, tmp_vec, visited, reset, dag, nr_sp):
    """
    Perform a simple breadth-first search (BFS) with path tracking.

    Parameters
    ----------
    C : int
        Maximum allowed cost (or distance).
    N : int
        Number of nodes in the network.
    bond : list
        Adjacency-like structure representing the graph.
    source : int
        Starting node.
    target : int
        Target node.
    vec, tmp_vec : list
        Temporary arrays for BFS layers.
    visited : list
        Marks visited nodes and their distances.
    reset : list
        Keeps track of nodes that must be reset after BFS.
    dag : list
        Directed acyclic graph structure to store predecessors.
    nr_sp : list
        Number of shortest paths reaching each node.

    Returns
    -------
    int
        0 if search ended without reaching max cost or target,
        1 if max cost was reached,
        2 if the target node was found.
    """
    int_distance = 0
    control = 0

    vec[0] = 1
    vec[1] = source
    visited[source] = 0
    dag[0][source] = 0
    nr_sp[source] = 1

    reset[0] = 1
    reset[1] = source

    while vec[0] > 0 and control == 0:
        tmp_vec[0] = 0
        int_distance += 1

        if int_distance == C:
            control = 1  # reached maximum cost

        for i in range(1, vec[0] + 1):
            n = vec[i]
            for j in range(1, bond[0][n] + 1):
                m = bond[n][j]

                if m == target:
                    control = 2  # target found

                if visited[m] < 0:
                    tmp_vec[0] += 1
                    tmp_vec[tmp_vec[0]] = m
                    visited[m] = visited[n] + 1
                    reset[0] += 1
                    reset[reset[0]] = m

                if visited[m] == int_distance:
                    dag[0][m] += 1
                    dag[m][dag[0][m]] = n
                    nr_sp[m] += nr_sp[n]

        for i in range(tmp_vec[0] + 1):
            vec[i] = tmp_vec[i]

    return control


def sample_random_path(C, N, dag, source, target, path, nr_sp):
    """
    Sample one random shortest path from source to target using BFS results.

    Parameters
    ----------
    C : int
        Maximum allowed cost.
    N : int
        Number of nodes.
    dag : list
        Directed acyclic graph storing predecessors.
    source : int
        Starting node.
    target : int
        Target node.
    path : list
        List to store the sampled path.
    nr_sp : list
        Number of shortest paths reaching each node.
    """
    control = 0
    path[0] = 1
    path[1] = target

    while control == 0:
        n = path[path[0]]
        if dag[0][n] > 0:
            T = 0
            for i in range(1, dag[0][n] + 1):
                T += nr_sp[dag[n][i]]

            q = int(genrand64_real3() * T) + 1
            if q > T:
                q = 1

            i = 0
            acc = 0
            found = 0

            while found == 0:
                i += 1
                acc += nr_sp[dag[n][i]]
                if q <= acc:
                    found = dag[n][i]

            path[0] += 1
            path[path[0]] = found

            if path[0] == C + 1 or path[path[0]] == source:
                control = 1
        else:
            path[0] = 0
            return


def get_path_of_pair(source, target, C, N, bond, path, vec, tmp_vec, visited, reset, dag, nr_sp):
    """
    Find one path between a source and target using BFS and random sampling.

    Parameters
    ----------
    source : int
        Starting node.
    target : int
        Target node.
    C : int
        Maximum allowed cost.
    N : int
        Number of nodes.
    bond : list
        Graph adjacency-like structure.
    path : list
        List to store the final path.
    vec, tmp_vec, visited, reset, dag, nr_sp : list
        Auxiliary arrays for BFS and path sampling.

    Returns
    -------
    int
        Length of the found path (0 if no path was found).
    """
    control = simple_bfs(C, N, bond, source, target, vec, tmp_vec, visited, reset, dag, nr_sp)
    path[0] = 0
    if control == 2:
        sample_random_path(C, N, dag, source, target, path, nr_sp)

    # Reset all visited nodes and arrays
    for i in range(1, reset[0] + 1):
        node = reset[i]
        visited[node] = -1
        dag[0][node] = 0
        nr_sp[node] = 0

    return path[0]


### 🎯 Geometric Distribution (Random Sampling)


In [5]:
def geometric_distribution(P):
    """
    Draw a random value from a geometric distribution.

    The geometric distribution models the number of trials needed 
    until the first success in a sequence of independent Bernoulli trials.

    Parameters
    ----------
    P : float
        Probability of success in each trial (0 < P <= 1).

    Returns
    -------
    int
        Number of trials until the first success.
        Returns -1 if P <= 0.
        Returns 1 if P == 1 (success always on the first trial).
    """
    if P <= 0:
        return -1
    if P == 1:
        return 1
    u = genrand64_real3()
    interval = int(1 + math.floor(math.log(u) / math.log(1. - P)))
    return interval


### 🔄 Pair Adjustment


In [ ]:
def adjust_pair(pairs, q):
    """
    Adjust the list of node pairs by removing one entry.

    This function swaps the pair at position q with the last available pair
    and then decreases the total count, effectively "removing" the chosen pair.

    Parameters
    ----------
    pairs : list of lists
        A 2D structure where:
        - pairs[0] stores the first node of each pair
        - pairs[1] stores the second node of each pair
        - pairs[0][0] keeps track of the current number of pairs.
    q : int
        Index of the pair to be removed/adjusted.

    Notes
    -----
    - This is a low-level operation adapted from C implementations.
    - It avoids shifting all elements by swapping with the last one.
    """
    node1 = pairs[0][q]
    node2 = pairs[1][q]  

    pairs[0][q] = pairs[0][pairs[0][0]]
    pairs[1][q] = pairs[1][pairs[0][0]]

    pairs[0][pairs[0][0]] = node1
    pairs[1][pairs[0][0]] = node2

    pairs[0][0] -= 1


### 🗑️ Pair Removal (With and Without Pair Info)

In [7]:
def remove_pair_w_pair_info_v2(C, N, bond, pairs, edges, update_time):
    """
    Remove pairs of nodes when pair information is available.

    This function iteratively removes node pairs from the network,
    updating adjacency structures, edge lists, and timestamps.

    Parameters
    ----------
    C : int
        Maximum allowed cost (or -1 for unlimited).
    N : int
        Number of nodes.
    bond : list
        Graph adjacency-like structure.
    pairs : list of lists
        List of node pairs available for removal.
    edges : list of lists
        Edge list to update with removed connections.
    update_time : list
        Timestamps updated as removals occur.
    """
    path = [0]*(N+1)
    vec = [0]*(N+1)
    tmp_vec = [0]*(N+1)
    visited = [-1]*(N+1)
    reset = [0]*(N+1)
    nr_sp = [0]*(N+1)

    dag = []
    for i in range(N+1):
        if i == 0:
            dag.append([0]*(N+1))
        else:
            dag.append([0]*(bond[0][i]+1))

    for i in range(1, N+1):
        visited[i] = -1
        dag[0][i] = 0
        nr_sp[i] = 0

    interval = 0
    E = edges[0][0]

    while pairs[0][0] > 0:
        num_pairs = pairs[0][0]
        link_prob = 2.0 * float(num_pairs)/(float(N)*float(N - 1))
        interval += geometric_distribution(link_prob)

        q = int(genrand64_real3()*num_pairs + 1)
        n = pairs[0][q]
        m = pairs[1][q]

        source = n
        target = m
        if bond[0][source] > bond[0][target]:
            n, m = m, n

        if C == -1:
            p = get_path_of_pair(n, m, N, N, bond, path, vec, tmp_vec,
                                 visited, reset, dag, nr_sp)
        else:
            p = get_path_of_pair(n, m, C, N, bond, path, vec, tmp_vec,
                                 visited, reset, dag, nr_sp)

        if p > 0:
            for j in range(1, path[0]):
                s = path[j]
                t = path[j + 1]

                v = find_node(s, t, bond)
                bond[s][v] = bond[s][bond[0][s]]
                bond[0][s] -= 1

                v = find_node(t, s, bond)
                bond[t][v] = bond[t][bond[0][t]]
                bond[0][t] -= 1

                E += 1
                if s < t:
                    edges[0][E] = s
                    edges[1][E] = t
                else:
                    edges[0][E] = t
                    edges[1][E] = s
                update_time[E] += interval

            control = simple_bfs(C, N, bond, n, m, vec, tmp_vec,
                                 visited, reset, dag, nr_sp)
            if control == 1:
                adjust_pair(pairs, q)
        else:
            adjust_pair(pairs, q)

        for i in range(1, reset[0] + 1):
            node = reset[i]
            visited[node] = -1
            dag[0][node] = 0
            nr_sp[node] = 0

    edges[0][0] = E


def remove_pair_wo_pair_info_v2(C, N, bond, edges, E, update_time, threshold):
    """
    Remove pairs of nodes when pair information is not available.

    This function randomly samples node pairs and attempts to remove them.
    The process continues until a failure threshold is reached.

    Parameters
    ----------
    C : int
        Maximum allowed cost (or -1 for unlimited).
    N : int
        Number of nodes.
    bond : list
        Graph adjacency-like structure.
    edges : list of lists
        Edge list to update with removed connections.
    E : int
        Initial number of edges.
    update_time : list
        Timestamps updated as removals occur.
    threshold : int
        Maximum number of consecutive failed removal attempts allowed.
    """
    path = [0]*(N+1)
    vec = [0]*(N+1)
    tmp_vec = [0]*(N+1)
    visited = [-1]*(N+1)
    reset = [0]*(N+1)
    nr_sp = [0]*(N+1)

    dag = []
    dag.append([0]*(N+1))
    for i in range(1, N+1):
        dag.append([0]*(bond[0][i]+1))
        visited[i] = -1
        dag[0][i] = 0
        nr_sp[i] = 0

    count_E = 0
    tau = 1
    num_failures = 0

    while num_failures < threshold:
        while True:
            n = int(genrand64_real3()*N) + 1
            m = int(genrand64_real3()*N) + 1
            if n != m:
                break

        source = n
        target = m
        if bond[0][source] > bond[0][target]:
            n, m = m, n

        if C == -1:
            p = get_path_of_pair(n, m, N, N, bond, path, vec, tmp_vec,
                                 visited, reset, dag, nr_sp)
        else:
            p = get_path_of_pair(n, m, C, N, bond, path, vec, tmp_vec,
                                 visited, reset, dag, nr_sp)

        if p > 0:
            num_failures = 0
            for j in range(1, path[0]):
                s = path[j]
                t = path[j + 1]

                v1 = find_node(s, t, bond)
                bond[s][v1] = bond[s][bond[0][s]]
                bond[0][s] -= 1

                v2 = find_node(t, s, bond)
                bond[t][v2] = bond[t][bond[0][t]]
                bond[0][t] -= 1

                count_E += 1
                if s < t:
                    edges[0][count_E] = s
                    edges[1][count_E] = t
                else:
                    edges[0][count_E] = t
                    edges[1][count_E] = s
                update_time[count_E] += tau
        else:
            num_failures += 1

        tau += 1

    # Adjust timestamps of non-removed edges
    for i in range(count_E + 1, E + 1):
        update_time[i] += tau

    edges[0][0] = count_E


### 🤝 Finding All Possible Node Pairs

In [8]:
def find_all_possible_pairs(C, N, bond, pairs):
    """
    Find all possible pairs of nodes connected by paths.

    This function uses BFS (breadth-first search) starting from each node
    to discover which node pairs can be connected. The resulting list of
    pairs is stored in the `pairs` structure.

    Parameters
    ----------
    C : int
        Maximum allowed cost (or -1 for unlimited).
    N : int
        Number of nodes in the graph.
    bond : list
        Graph adjacency-like structure.
    pairs : list of lists
        2D structure where pairs[0] and pairs[1] store the node indices
        of each pair.

    Returns
    -------
    int
        Total number of pairs found.

    Notes
    -----
    - Ensures that each pair (i, j) is stored only once (i < j).
    - Expands `pairs` dynamically if needed.
    """
    num_pairs = 0
    dag = []
    list_pairs = []

    vec = [0]*(N+1)
    tmp_vec = [0]*(N+1)
    visited = [-1]*(N+1)
    reset = [0]*(N+1)
    nr_sp = [0]*(N+1)

    dag.append([0]*(N+1))
    for i in range(1, N+1):
        dag.append([0]*(bond[0][i]+1))

    list_pairs.append([0]*(N+1))
    for i in range(1, N+1):
        list_pairs.append([])
        list_pairs[i].append(0)

    for i in range(1, N+1):
        visited[i] = -1
        dag[0][i] = 0
        nr_sp[i] = 0

    target = -1

    for source in range(1, N+1):
        control = simple_bfs(C, N, bond, source, target,
                             vec, tmp_vec, visited, reset, dag, nr_sp)

        for k in range(1, reset[0] + 1):
            n = reset[k]
            if source < n and find_node(source, n, list_pairs) < 0:
                list_pairs[0][source] += 1
                list_pairs[source].append(n)
                num_pairs += 1

        for k in range(1, reset[0] + 1):
            node = reset[k]
            visited[node] = -1
            dag[0][node] = 0
            nr_sp[node] = 0

    # Expand pairs array if necessary
    while len(pairs[0]) < num_pairs + 1:
        pairs[0].append(0)
    while len(pairs[1]) < num_pairs + 1:
        pairs[1].append(0)

    num_pairs = 0
    for i in range(1, N+1):
        for j in range(1, list_pairs[0][i] + 1):
            num_pairs += 1
            pairs[0][num_pairs] = i
            pairs[1][num_pairs] = list_pairs[i][j]

    pairs[0][0] = num_pairs

    return num_pairs


### 🌳 Tree Functions (Finding Roots)

In [9]:
def tree_find_root(n, root, res):
    """
    Find the root of a node in a tree-like structure.

    This function recursively follows parent pointers until it finds
    the root of the given node.

    Parameters
    ----------
    n : int
        Node index to find the root for.
    root : list of lists
        Tree representation where:
        - root[0][i] gives the parent of node i
        - root[1][i] can store additional info (e.g., depth or weight).
    res : list
        Output list of size 2 where:
        - res[0] = root node index
        - res[1] = associated info from root[1].

    Notes
    -----
    - If root[0][n] == n, then `n` is its own root.
    - This is a recursive implementation, similar to union-find
      structures without path compression.
    """
    if root[0][n] == n:
        res[0] = root[0][n]
        res[1] = root[1][n]
        return
    tree_find_root(root[0][n], root, res)


### 🧩 Modified Newman-Ziff Algorithm (Cluster Growth)

In [10]:
def modified_NZ_algorithm(N, edges, largest_cluster):
    """
    Modified Newman-Ziff algorithm for tracking cluster growth.

    This algorithm processes edges in reverse order to merge clusters
    and measure the size of the largest connected component as the 
    network evolves.

    Parameters
    ----------
    N : int
        Number of nodes in the network.
    edges : list of lists
        Edge list. 
        - edges[0][0] stores the number of edges.
        - edges[0][e], edges[1][e] represent an edge (n, m).
    largest_cluster : list of lists
        Data structure to store statistics about the largest cluster.
        For each edge e:
        - largest_cluster[1][e] = relative size of largest cluster
        - largest_cluster[2][e] = (size)^2
        - largest_cluster[3][e] = (size)^3
        - largest_cluster[4][e] = (size)^4
        - largest_cluster[5][e] = adjusted variance

    Notes
    -----
    - Uses a union-find–like structure (`root`) to track connected components.
    - Calls `tree_find_root` to locate roots of nodes before merging.
    - Updates running statistics of the largest cluster as edges are added back.
    """
    root = [ [0]*(N+1), [0]*(N+1) ]
    res = [0, 0]

    larg = 1
    av2 = float(N - 1)

    # initialize root structure
    for i in range(1, N+1):
        root[0][i] = i
        root[1][i] = 1

    E = edges[0][0]
    for e in range(E, 0, -1):
        n = edges[0][e]
        m = edges[1][e]

        # av2 += larg^2
        av2 += float(larg) * float(larg)

        # find roots of n and m
        tree_find_root(n, root, res)
        rn, sn = res[0], res[1]

        tree_find_root(m, root, res)
        rm, sm = res[0], res[1]

        if rn != rm:
            av2 -= float(sn) * float(sn)
            av2 -= float(sm) * float(sm)

            if sn > sm:
                root[0][rm] = rn
                root[1][rm] = 1
                root[1][rn] = sn + sm
            else:
                root[0][rn] = rm
                root[1][rn] = 1
                root[1][rm] = sn + sm

            if sn + sm > larg:
                larg = sn + sm

        tmp = float(larg) / float(N)
        largest_cluster[1][e] = tmp
        largest_cluster[2][e] = tmp * tmp
        largest_cluster[3][e] = tmp * tmp * tmp
        largest_cluster[4][e] = tmp * tmp * tmp * tmp

        av = float(N) - float(larg)
        if rn != rm:
            av2 += float(sn + sm) * float(sn + sm)
        av2 -= float(larg) * float(larg)
        if av > 0:
            largest_cluster[5][e] = av2 / av


### ⚡ Efficient Pair Removal 

In [11]:
def efficient_pair_removal(C, N, bond, edges, E, update_time, threshold):
    """
    Efficient procedure to remove node pairs in two phases.

    Phase 1: Removes pairs without explicit pair information 
             (random sampling until a threshold of failures).
    Phase 2: Identifies all possible pairs and removes them using 
             explicit pair information.

    Parameters
    ----------
    C : int
        Maximum allowed cost (or -1 for unlimited).
    N : int
        Number of nodes in the graph.
    bond : list
        Graph adjacency-like structure.
    edges : list of lists
        Edge list storing current network connections.
    E : int
        Initial number of edges.
    update_time : list
        Array that records timestamps for each edge removal.
    threshold : int
        Maximum number of consecutive failed removals allowed 
        during Phase 1.

    Notes
    -----
    - Combines `remove_pair_wo_pair_info_v2` (Phase 1) with 
      `find_all_possible_pairs` + `remove_pair_w_pair_info_v2` (Phase 2).
    - This mimics the structure of the C implementation, 
      but in Python memory allocation is simpler (no malloc/free).
    """
    # Phase 1: remove pairs without explicit pair info
    remove_pair_wo_pair_info_v2(C, N, bond, edges, E, update_time, threshold)

    # Phase 2: generate all possible pairs and remove them
    pairs = [ [0], [0] ]  # Python replacement for malloc
    num_pairs = find_all_possible_pairs(C, N, bond, pairs)
    remove_pair_w_pair_info_v2(C, N, bond, pairs, edges, update_time)


### 🏁 Main Simulation Routine

In [12]:
def main(N=1024, avg_k=4, C=None, num_iter=10, num_instance=1, output_dir="./data_notebook"):
    """
    Main function to run the Shortest Path Percolation (SPP) simulation.

    Parameters
    ----------
    N : int, default=1024
        Number of nodes in the network.
    avg_k : int, default=4
        Mean degree of the network.
    C : int or None, default=None
        Maximum allowed cost. If None, defaults to N.
    num_iter : int, default=10
        Number of iterations per instance.
    num_instance : int, default=1
        Number of independent instances.
    output_dir : str, default="./data_notebook"
        Directory where results will be saved.

    Notes
    -----
    - Designed for Jupyter/Colab usage, no need for sys.argv.
    - Saves results in CSV files under the chosen output_dir.
    """
    start = time.time()

    if C is None:
        C = N

    E = int(0.5 * N * avg_k)

    # Allocate structures
    bond = [[] for _ in range(N+1)]
    bond[0] = [0] * (N+1)
    for i in range(1, N+1):
        bond[i] = [0]

    copy_bond = [[] for _ in range(N+1)]
    copy_bond[0] = [0] * (N+1)

    edges = [[] for _ in range(2)]
    edges[0] = [0] * (E+1)
    edges[1] = [0] * (E+1)

    update_time = [0] * (E+1)

    largest_cluster = []
    for _ in range(6):
        largest_cluster.append([0.0] * (E+1))

    # Seed for random generator
    pid_id = int(time.time()) * os.getpid()
    init_genrand64(pid_id)

    for x in range(num_instance):
        generate_ERgraph(N, bond, E)

        for i in range(num_iter):
            copy_network(N, bond, copy_bond)
            clean_time_stamp(update_time, E)

            # threshold = 7*N^0.44
            threshold = int(7 * (N ** 0.44))

            # Run efficient pair removal
            efficient_pair_removal(C, N, copy_bond, edges, E, update_time, threshold)

            # Run modified Newman-Ziff algorithm
            modified_NZ_algorithm(N, edges, largest_cluster)

            # Output directory and file name
            if C > 0:
                dir_path = f"{output_dir}/N{N}/k{avg_k}/C{C}/"
            else:
                dir_path = f"{output_dir}/N{N}/k{avg_k}/C{N}/"

            file_id = i
            file_name = f"{dir_path}full_data_{file_id}.csv"
            os.makedirs(dir_path, exist_ok=True)

            # Save results to CSV
            with open(file_name, "w") as f:
                for j in range(1, edges[0][0] + 1):
                    removed_edges = float(j) / float(edges[0][0])
                    f.write(f"{removed_edges:.10f} "
                            f"{largest_cluster[1][j]:.10f} "
                            f"{largest_cluster[2][j]:.10f} "
                            f"{largest_cluster[3][j]:.10f} "
                            f"{largest_cluster[4][j]:.10f} "
                            f"{largest_cluster[5][j]:.10f} "
                            f"{edges[0][j]} "
                            f"{edges[1][j]} "
                            f"{update_time[j]}\n")

    end = time.time()
    cpu_time_used = (end - start)
    print(f"# mean simulation time for N={N} C={C} "
          f"calculated from {num_instance} instances and {num_iter} iterations: "
          f"{cpu_time_used/(num_instance*num_iter):.4f} seconds")


## 🏁 Running the Simulation

The `main()` function runs the **Shortest Path Percolation (SPP) simulation**.  
Its main inputs are:

- **N**: number of nodes in the network (e.g., 4096).  
- **avg_k**: average degree of the network (e.g., 4).  
- **C**: maximum allowed cost for paths (use `N` or `-1` for unlimited).  
- **num_iter**: number of iterations per instance (repeated runs for statistics).  
- **num_instance**: number of independent instances of the experiment.  

Example:
```python
main(N=4096, avg_k=4, C=4096, num_iter=10, num_instance=1)


In [29]:
main(N=4096, avg_k=4, C=4096, num_iter=10, num_instance=1)

# mean simulation time for N=4096 C=4096 calculated from 1 instances and 10 iterations: 1.1633 seconds


----------------
----------------